# Model III
## Rozpoznawanie instrumentów za pomocą konwolucji (ResNet18)

### Biblioteki

In [12]:
import sys
import os
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

import time

from model import Resnet18
from trening import get_device, train_model
from przygotowanie_danych import (
    create_datasets,
    create_loaders
)

Hiperparametry modelu

In [14]:
dataset = "1_3"
model_name = "resnet18"

epochs = 100
patience = 10
batch_size = 16
learning_rate = 0.001
dropout = 0.1

weights_type = "default"
weights_path = None

In [15]:
results_file = Path("wyniki/modele.csv")

if results_file.exists():
    results = pd.read_csv(results_file)

    combination_exists = (
        (results["Dataset"] == dataset) &
        (results["Model"] == model_name) &
        (results["Batch size"] == batch_size) &
        (results["Learning rate"] == learning_rate) &
        (results["Dropout"] == dropout)
    ).any()

    if combination_exists:
        raise ValueError("Ta kombinacja datasetu i hiperparametrów była już testowana. Zatrzymuję program.")

print("Nowa kombinacja. Kontynuuję wykonywanie kodu.")

Nowa kombinacja. Kontynuuję wykonywanie kodu.


In [16]:
existing_numbers = [
    int(f[:-4])
    for f in os.listdir("wyniki")
    if f.endswith(".pth")
]

learning_id = max(existing_numbers, default=0) + 1

weights_filename = f"{learning_id}.pth"

weights_filename

'31.pth'

In [17]:
train_dataset, val_dataset, test_dataset = create_datasets(dataset)

train_loader, val_loader, test_loader = create_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

Wczytuję spektrogramy z: C:\Users\Wiktor\Documents\GitHub\instrument-recognition-project\spectrograms\1_3


In [ ]:
device = get_device()
print(device)

model = Resnet18(
    dropout=dropout,
    weights_type=weights_type,
    weights_path=weights_path
).to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

cuda


Pętla treningowa

In [19]:
start_time = time.perf_counter()

model, history, best_epoch, best_val_loss = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    patience=patience,
    model_name= str(weights_filename)
)

training_time = time.perf_counter() - start_time

print(f"Czas treningu: {training_time / 60:.2f} min")

Epoka 1/100 | Train loss: 0.4076 | Val loss: 0.4887
Epoka 2/100 | Train loss: 0.2586 | Val loss: 0.3770
Epoka 3/100 | Train loss: 0.1942 | Val loss: 0.2982
Epoka 4/100 | Train loss: 0.1565 | Val loss: 0.3908
Epoka 5/100 | Train loss: 0.1282 | Val loss: 0.3197
Epoka 6/100 | Train loss: 0.1040 | Val loss: 0.3940
Epoka 7/100 | Train loss: 0.0886 | Val loss: 0.3712
Epoka 8/100 | Train loss: 0.0763 | Val loss: 0.4453
Epoka 9/100 | Train loss: 0.0674 | Val loss: 0.4309
Epoka 10/100 | Train loss: 0.0603 | Val loss: 0.3603
Epoka 11/100 | Train loss: 0.0526 | Val loss: 0.3982
Epoka 12/100 | Train loss: 0.0437 | Val loss: 0.3816
Epoka 13/100 | Train loss: 0.0386 | Val loss: 0.4015
Brak poprawy przez 10 epok.
Najlepsza epoka: 3
Najlepszy validation loss: 0.2982
Czas treningu: 8.39 min


Zapisanie danych o modelu

In [20]:
experiment = pd.DataFrame([{
    "Learning_ID": learning_id,
    "Dataset": dataset,
    "Model": model_name,
    "Batch size": batch_size,
    "Learning rate": learning_rate,
    "Dropout": dropout,
    "Best epoch": best_epoch,
    "Best val loss": best_val_loss,
    "Training time [min]": training_time / 60
}])

experiment.to_csv(
    results_file,
    mode="a",
    header=not results_file.exists(),
    index=False
)

In [21]:
history_df = pd.DataFrame(history)

history_df.to_csv(
    f"wyniki/history_{learning_id}.csv",
    index_label="epoch"
)